# InSAR deformation profile: 2023 Türkiye–Syria earthquake (Mw 7.8)

Co-seismic deformation from a Sentinel-1 interferogram (COMET-LiCSAR, 29 Jan 2023 to 10 Feb 2023) that spans the 6 February 2023 earthquakes. The notebook loads wrapped phase, unwrapped phase and coherence, extracts a phase profile across the deformation zone with WGS84 geodesic distances, and masks low-coherence samples.

Input GeoTIFFs go in `data/` (see the README). Figures are written to `figures/`.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.transform import xy
from pyproj import Geod

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

PAIR = "20230129_20230210"  # LiCSAR interferogram: primary_secondary date
PHASE_FILE = DATA_DIR / f"{PAIR}.geo.diff_pha.tif"  # wrapped phase
UNW_FILE = DATA_DIR / f"{PAIR}.geo.unw.tif"         # unwrapped phase
COH_FILE = DATA_DIR / f"{PAIR}.geo.cc.tif"          # coherence

# Profile end points as (column, row) in pixel coordinates, placed across
# the main fringe pattern and roughly perpendicular to the fault
START = (2500, 4300)
END = (1750, 600)
N_POINTS = 2000

COH_THRESHOLD = 0.2  # profile samples below this coherence are masked

## 1. Load the interferogram

In [ ]:
def load_geotiff(path):
    with rasterio.open(path) as src:
        return src.read(1), src.profile


phase, phase_meta = load_geotiff(PHASE_FILE)

coh_raw, _ = load_geotiff(COH_FILE)
# LiCSAR stores coherence as 8-bit integers (0-255); scale to 0-1
coh = coh_raw / 255.0 if coh_raw.dtype == np.uint8 else coh_raw.astype(float)

if UNW_FILE.exists():
    unw, _ = load_geotiff(UNW_FILE)
else:
    unw = None
    print("No unwrapped phase file found; unwrapped plots are skipped.")

print("Raster size (rows, cols):", phase.shape)

## 2. Wrapped phase and coherence

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(phase, cmap="jet")
ax.set_title("Wrapped phase")
fig.colorbar(im, ax=ax, label="Phase (rad)")
fig.savefig(FIG_DIR / "wrapped_phase.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(coh, vmin=0, vmax=1, cmap="jet")
ax.set_title("Coherence")
fig.colorbar(im, ax=ax, label="Coherence")
fig.savefig(FIG_DIR / "coherence_map.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Profile across the deformation zone

In [ ]:
cols = np.linspace(START[0], END[0], N_POINTS).astype(int)
rows = np.linspace(START[1], END[1], N_POINTS).astype(int)

phase_line = phase[rows, cols]
coh_line = coh[rows, cols]

# Pixel centres to longitude/latitude, then WGS84 geodesic distance along the line
lon, lat = xy(phase_meta["transform"], rows, cols)
lon, lat = np.asarray(lon), np.asarray(lat)

geod = Geod(ellps="WGS84")
_, _, segment_m = geod.inv(lon[:-1], lat[:-1], lon[1:], lat[1:])
distance_km = np.concatenate([[0.0], np.cumsum(segment_m)]) / 1000

print(f"Profile length: {distance_km[-1]:.1f} km")

## 4. Coherence mask

In [ ]:
good = coh_line >= COH_THRESHOLD
print(f"Coherence threshold: {COH_THRESHOLD}")
print(f"Samples kept: {good.sum()} of {good.size} ({good.size - good.sum()} removed)")

fig, ax1 = plt.subplots(figsize=(15, 3))
ax1.plot(distance_km[good], phase_line[good], ".", color="blue", markersize=3, label="Phase")
ax1.set_xlabel("Distance along profile (km)")
ax1.set_ylabel("Phase (rad)", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(distance_km[good], coh_line[good], ".", color="red", markersize=2, label="Coherence")
ax2.set_ylabel("Coherence", color="red")
ax2.tick_params(axis="y", labelcolor="red")
ax2.set_ylim(0, 1)

ax1.set_title(f"Profile after coherence masking (coherence ≥ {COH_THRESHOLD})")
handles = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
ax1.legend(handles, [h.get_label() for h in handles], loc="upper center",
           bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
fig.savefig(FIG_DIR / "filtered_profile.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Wrapped and unwrapped profiles

In [ ]:
def draw_profile_line(ax):
    ax.plot([START[0], END[0]], [START[1], END[1]], "k-", linewidth=2)
    ax.set_xlabel("Pixel")
    ax.set_ylabel("Pixel")


def profile_panel(ax, dist, values, coherence, color, title):
    ax.plot(dist, values, ".", color=color, markersize=2)
    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("Phase (rad)", color=color)
    ax.tick_params(axis="y", labelcolor=color)
    ax.grid(True)
    ax.set_title(title)
    axc = ax.twinx()
    axc.plot(dist, coherence, "--", color="red", alpha=0.6)
    axc.set_ylabel("Coherence", color="red")
    axc.tick_params(axis="y", labelcolor="red")
    axc.set_ylim(0, 1)


if unw is None:
    print("No unwrapped phase available.")
else:
    unw_line = unw[rows, cols]
    valid = ~np.isnan(unw_line) & (unw_line != 0)  # 0 marks no data in LiCSAR products

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    im = axes[0, 0].imshow(phase, cmap="jet")
    draw_profile_line(axes[0, 0])
    axes[0, 0].set_title("Wrapped phase")
    fig.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04, label="Phase (rad)")

    im = axes[0, 1].imshow(unw, cmap="jet")
    draw_profile_line(axes[0, 1])
    axes[0, 1].set_title("Unwrapped phase")
    fig.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04, label="Phase (rad)")

    profile_panel(axes[1, 0], distance_km, phase_line, coh_line, "blue", "Wrapped profile")
    profile_panel(axes[1, 1], distance_km[valid], unw_line[valid], coh_line[valid],
                  "green", "Unwrapped profile")

    fig.suptitle("InSAR profile analysis: wrapped vs unwrapped", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(FIG_DIR / "wrapped_vs_unwrapped_profile.png", dpi=150, bbox_inches="tight")
    plt.show()